# Project 3: EEG-Vision-XAI (Kaggle version)

Same experiment as the Colab notebook: spectrogram images, ImageNet ResNet-18 / EfficientNet-B0 / scratch / linear probe,
**all 23 CHB-MIT subjects**, subject-wise 5-fold CV, Grad-CAM and occlusion XAI. **It does not need Project 1 or 2 results.**

**Before running (right-hand panel → Settings):**
1. **Accelerator → GPU T4 x2.** P100 is not supported by recent PyTorch builds.
2. **Internet → On.** This needs a phone-verified Kaggle account.
3. Then click **Save Version → Save & Run All (Commit)**. It runs in the background (up to 12 h), so you can close the browser.

**Time:** about 30–60 min of data preparation plus 4–7 h of training. There is a built-in 10.5 h budget, so results are always saved before
Kaggle's 12 h limit. **Results:** the version's *Output* tab → `project3_eeg_vision_xai/` and `project3_eeg_vision_xai_results.zip`.

**If it didn't finish:** open the notebook, *Add Input → Your Work → this notebook's previous version*, and Save & Run All again.
It continues from the saved `results.json`.

In [ ]:
# ---- 1. Setup for Kaggle: GPU + internet check, output folders ----
import os, sys, subprocess, shutil, glob, urllib.request
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. In the right panel: Settings -> Accelerator -> 'GPU T4 x2', then run again.")
_p = torch.cuda.get_device_properties(0)
print(f"GPU: {_p.name}, {_p.total_memory/2**30:.1f} GB, compute capability {_p.major}.{_p.minor}")
if _p.major < 7:
    print("WARNING: P100 (Pascal) is not supported by recent PyTorch builds. Prefer Accelerator -> 'GPU T4 x2'.")
try:
    urllib.request.urlopen("https://physionet-open.s3.amazonaws.com/chbmit/1.0.0/RECORDS", timeout=20)
    print("internet: OK")
except Exception:
    raise SystemExit("No internet. Right panel: Settings -> Internet -> On (needs a phone-verified Kaggle account).")
OUT = Path('/kaggle/working/project3_eeg_vision_xai_v2')   # saved as notebook output
OUT.mkdir(parents=True, exist_ok=True)
# Resume: if you attached the output of a previous run as an input, continue from its results
for prev in glob.glob('/kaggle/input/**/project3_eeg_vision_xai_v2/results.json', recursive=True):
    src = Path(prev).parent
    shutil.copytree(src, OUT, dirs_exist_ok=True); print('resuming from previous run:', src); break
WORK = Path('/kaggle/temp'); WORK.mkdir(parents=True, exist_ok=True)   # big temporary data (not saved)
os.chdir('/kaggle/working'); sys.path.insert(0, '/kaggle/working')
print('results folder:', OUT)

## 2. Source code

In [ ]:
%%writefile eegkit.py
"""eegkit: shared EEG data utilities for the three Colab notebooks.

* Downloads CHB-MIT (24 cases / 23 subjects) and Siena Scalp EEG from PhysioNet's S3 mirror
* Converts every recording to the same 18-channel longitudinal bipolar montage
  (works for bipolar CHB-MIT files and referential Siena files)
* 0.5-45 Hz band-pass, 60/50 Hz notch, resampling to 128 Hz, float16 cache
* Windowing, subject-wise folds, metrics
"""
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
from scipy import signal

MONTAGE = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ",
]
FS = 128
S3 = "https://physionet-open.s3.amazonaws.com"
CHB = f"{S3}/chbmit/1.0.0"
SIENA = f"{S3}/siena-scalp-eeg/1.0.0"
OLD_NAMES = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}


# ----------------------------------------------------------------------------- io
def fetch_text(url: str) -> str:
    for _ in range(5):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return r.read().decode("latin-1")
        except Exception:  # noqa: BLE001
            pass
    raise RuntimeError(f"could not fetch {url}")


def download(url: str, dst: Path) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    tmp = dst.with_suffix(dst.suffix + ".part")
    subprocess.run(["curl", "-sfL", "--retry", "5", "-o", str(tmp), url.replace("+", "%2B")], check=True)
    tmp.rename(dst)
    return dst


def subject_of(case: str) -> str:
    """chb21 is the same subject as chb01 (recorded 1.5 years later)."""
    return "chb01" if case == "chb21" else case


# ---------------------------------------------------------------------- metadata
def parse_chb_summary(text: str) -> dict[str, list[tuple[float, float]]]:
    recs, cur, starts = {}, None, []
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("File Name:"):
            cur = line.split(":", 1)[1].strip(); recs[cur] = []; starts = []
        elif cur and re.match(r"Seizure\s*\d*\s*Start Time", line):
            starts.append(float(re.findall(r"(\d+)\s*seconds", line)[0]))
        elif cur and re.match(r"Seizure\s*\d*\s*End Time", line):
            recs[cur].append((starts.pop(0), float(re.findall(r"(\d+)\s*seconds", line)[0])))
    return recs


def chbmit_catalog(raw_root: Path) -> list[dict]:
    """All CHB-MIT recordings with seizure annotations."""
    raw_root.mkdir(parents=True, exist_ok=True)
    records = fetch_text(f"{CHB}/RECORDS").split()
    cat = []
    summaries = {}
    for rel in records:
        case, fname = rel.split("/")
        if case not in summaries:
            summaries[case] = parse_chb_summary(fetch_text(f"{CHB}/{case}/{case}-summary.txt"))
        cat.append({"dataset": "chbmit", "case": case, "subject": subject_of(case), "rel": rel,
                    "file": fname.replace(".edf", ".npy"),
                    "seizures": summaries[case].get(fname, [])})
    return cat


def siena_catalog() -> list[dict]:
    records = fetch_text(f"{SIENA}/RECORDS").split()
    return [{"dataset": "siena", "case": r.split("/")[0], "subject": r.split("/")[0], "rel": r,
             "file": r.split("/")[1].replace(".edf", ".npy"), "seizures": []} for r in records]


# ------------------------------------------------------------------ signal path
def _norm(label: str) -> str:
    l = label.upper().replace("EEG ", "").replace("-REF", "").replace("-LE", "").replace("-CS2", "").strip()
    return l


def read_edf(path: Path) -> tuple[list[str], list[float], list[np.ndarray]]:
    """Minimal pure-NumPy EDF/EDF+ reader (no compiled dependency).
    Returns labels, sampling rates and physical-unit signals (annotation channels skipped)."""
    with open(path, "rb") as f:
        h = f.read(256)
        n_hdr = int(h[184:192]); n_rec = int(h[236:244]); dur = float(h[244:252]); ns = int(h[252:256])
        e = f.read(ns * 256)

        def field(off, width, cast=str):
            base = off * ns
            return [cast(e[base + i * width: base + (i + 1) * width].decode("latin-1").strip()) for i in range(ns)]

        labels = field(0, 16)
        pmin, pmax = field(16 + 80 + 8, 8, float), field(16 + 80 + 8 + 8, 8, float)
        dmin, dmax = field(16 + 80 + 8 + 16, 8, float), field(16 + 80 + 8 + 24, 8, float)
        spr = field(16 + 80 + 8 + 32 + 80, 8, int)
        f.seek(n_hdr)
        raw = np.fromfile(f, dtype="<i2")
    rec_len = sum(spr)
    if n_rec <= 0:
        n_rec = len(raw) // rec_len
    raw = raw[: n_rec * rec_len].reshape(n_rec, rec_len)
    sigs, fss, labs, off = [], [], [], 0
    for i in range(ns):
        seg = raw[:, off: off + spr[i]].reshape(-1); off += spr[i]
        # skip annotation channels and CHB-MIT dummy separator channels ('-', digital range 0..0)
        if "annotation" in labels[i].lower() or dmax[i] == dmin[i] or labels[i].strip("-. ") == "":
            continue
        gain = (pmax[i] - pmin[i]) / (dmax[i] - dmin[i])
        sigs.append((seg.astype(np.float64) - dmin[i]) * gain + pmin[i])
        fss.append(spr[i] / dur); labs.append(labels[i])
    return labs, fss, sigs


def load_montage(path: Path) -> tuple[np.ndarray, float]:
    """Return (18, T) microvolts in MONTAGE order and the sampling rate.
    Bipolar labels are used directly; otherwise the pair is derived from referential channels."""
    raw_labels, fss, sigs = read_edf(path)
    labels = [_norm(l) for l in raw_labels]
    ref = {OLD_NAMES.get(l, l): i for i, l in enumerate(labels) if "-" not in l}
    out, fs = [], None
    for ch in MONTAGE:
        cands = [i for i, l in enumerate(labels) if l == ch or l.startswith(ch + "-")]
        if cands:
            i = cands[0]; out.append(sigs[i]); fs = fs or fss[i]; continue
        a, b = ch.split("-")
        if a in ref and b in ref:
            out.append(sigs[ref[a]] - sigs[ref[b]]); fs = fs or fss[ref[a]]; continue
        raise KeyError(f"{ch} not available in {path.name}")
    n = min(len(x) for x in out)
    return np.stack([x[:n] for x in out]), float(fs)


def preprocess(x: np.ndarray, fs: float, notch: float) -> np.ndarray:
    sos = signal.butter(4, [0.5, 45.0], btype="bandpass", fs=fs, output="sos")
    x = signal.sosfiltfilt(sos, x, axis=-1)
    b, a = signal.iirnotch(notch, 30.0, fs=fs)
    x = signal.filtfilt(b, a, x, axis=-1)
    from fractions import Fraction
    fr = Fraction(FS / fs).limit_denominator(64)
    x = signal.resample_poly(x, fr.numerator, fr.denominator, axis=-1)
    return x.astype(np.float32)


def _process_one(rec: dict, raw_root: Path, cache_root: Path, keep_raw: bool,
                 on_array=None) -> dict | None:
    out = cache_root / rec["file"] if cache_root is not None else None
    base = CHB if rec["dataset"] == "chbmit" else SIENA
    if out is not None and out.exists():
        rec = dict(rec, n_samples=int(np.load(out, mmap_mode="r").shape[1]))
        return rec
    raw = download(f"{base}/{rec['rel']}", raw_root / rec["rel"])
    try:
        x, fs = load_montage(raw)
        notch = 60.0 if rec["dataset"] == "chbmit" else 50.0
        x = preprocess(x, fs, notch)
    except Exception as e:  # noqa: BLE001
        print(f"  skip {rec['rel']}: {str(e)[:90]}", flush=True)
        if not keep_raw:
            raw.unlink(missing_ok=True)
        return None
    if on_array is not None:
        on_array(rec, x)
    if cache_root is not None:
        np.save(out, x.astype(np.float16))
    if not keep_raw:
        raw.unlink(missing_ok=True)
    return dict(rec, n_samples=int(x.shape[1]))


def build_cache(catalog: list[dict], raw_root: Path, cache_root: Path, workers: int = 4,
                keep_raw: bool = False) -> list[dict]:
    """Download + preprocess in parallel; writes cache_root/index.json. Resumable."""
    cache_root.mkdir(parents=True, exist_ok=True)
    done = []
    with ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(_process_one, r, raw_root, cache_root, keep_raw): r for r in catalog}
        for k, fu in enumerate(as_completed(futs)):
            r = fu.result()
            if r is not None:
                done.append(r)
            if (k + 1) % 20 == 0 or k + 1 == len(catalog):
                h = sum(d["n_samples"] for d in done) / FS / 3600
                print(f"  {k + 1}/{len(catalog)} recordings processed ({h:.0f} h cached)", flush=True)
    done.sort(key=lambda d: d["rel"])
    (cache_root / "index.json").write_text(json.dumps(done, indent=0))
    return done


def load_index(cache_root: Path) -> list[dict]:
    return json.loads((cache_root / "index.json").read_text())


def select_for_windows(catalog: list[dict], n_free_per_case: int = 3) -> list[dict]:
    """All seizure recordings + the first n seizure-free recordings of every case."""
    out, free = [], {}
    for r in catalog:
        if r["seizures"]:
            out.append(r)
        elif free.get(r["case"], 0) < n_free_per_case:
            free[r["case"]] = free.get(r["case"], 0) + 1; out.append(r)
    return out


# ------------------------------------------------------------------ windowing
def seizure_mask(n: int, seizures) -> np.ndarray:
    m = np.zeros(n, dtype=bool)
    for s, e in seizures:
        m[int(s * FS): int(e * FS)] = True
    return m


def make_windows(cache_root: Path, win_s=4.0, ictal_stride_s=1.0, inter_stride_s=8.0,
                 inter_ratio=10.0, guard_s=300.0, seed=0):
    """Labelled windows. Returns X float16 (N,18,T), y, subject, record-id."""
    rng = np.random.default_rng(seed)
    W = int(win_s * FS)
    index = load_index(cache_root)
    cands = []
    for r_id, rec in enumerate(index):
        n = rec["n_samples"]
        m = seizure_mask(n, rec["seizures"])
        g = seizure_mask(n, [(max(0, s - guard_s), e + guard_s) for s, e in rec["seizures"]])
        for s in range(0, n - W, int(ictal_stride_s * FS)):
            if m[s:s + W].mean() >= 0.5:
                cands.append((r_id, s, 1, rec["subject"]))
        for s in range(0, n - W, int(inter_stride_s * FS)):
            if not g[s:s + W].any():
                cands.append((r_id, s, 0, rec["subject"]))
    lab = np.array([c[2] for c in cands]); sub = np.array([c[3] for c in cands])
    keep = []
    for p in np.unique(sub):
        ii = np.where((sub == p) & (lab == 1))[0]; jj = np.where((sub == p) & (lab == 0))[0]
        if len(ii) == 0:
            continue
        keep.extend(ii); keep.extend(rng.choice(jj, min(len(jj), int(inter_ratio * len(ii))), replace=False))
    keep = np.sort(np.array(keep))
    X = np.empty((len(keep), len(MONTAGE), W), np.float16)
    mm = {}
    for k, i in enumerate(keep):
        r_id, s, _, _ = cands[i]
        if r_id not in mm:
            mm[r_id] = np.load(cache_root / index[r_id]["file"], mmap_mode="r")
        X[k] = mm[r_id][:, s:s + W]
    return X, lab[keep], sub[keep], np.array([cands[i][0] for i in keep])


def subject_folds(subjects: np.ndarray, y: np.ndarray, k: int = 5, seed: int = 0) -> list[list[str]]:
    """Greedy subject-wise k-fold that balances the number of ictal windows per fold."""
    subs = sorted(set(subjects))
    if k >= len(subs):
        return [[s] for s in subs]
    rng = np.random.default_rng(seed)
    ict = {s: int(((subjects == s) & (y == 1)).sum()) for s in subs}
    order = sorted(subs, key=lambda s: -ict[s] + rng.random() * 1e-3)
    folds, load = [[] for _ in range(k)], np.zeros(k)
    for s in order:
        j = int(np.argmin(load)); folds[j].append(s); load[j] += ict[s]
    return folds


# ---------------------------------------------------------------------- metrics
def window_metrics(y, p):
    from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
    fpr, tpr, _ = roc_curve(y, p)
    return {"auroc": float(roc_auc_score(y, p)), "auprc": float(average_precision_score(y, p)),
            "sens_at_spec95": float(tpr[np.searchsorted(fpr, 0.05, side="right") - 1])}


def drive_or_local(drive_dir: str, local_dir: str) -> Path:
    """Use Google Drive when mounted (Colab), else a local folder."""
    p = Path(drive_dir) if Path("/content/drive/MyDrive").exists() else Path(local_dir)
    p.mkdir(parents=True, exist_ok=True)
    return p


def copy_tree(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        if f.is_file() and not (dst / f.name).exists():
            shutil.copy2(f, dst / f.name)


if __name__ == "__main__":  # quick self-test of the montage loader
    import sys
    x, fs = load_montage(Path(sys.argv[1]))
    print(x.shape, fs, x.std(1)[:4])

In [ ]:
%%writefile eeg_image.py
"""Turn multichannel EEG into images on the GPU, and the CNN models used on them.

Image layout ("channel-stacked spectrogram"):
  rows  = 18 bipolar channels x 20 frequency rows (1-40 Hz, 2 Hz per row)
  cols  = 128 time columns covering the 8 s window (full STFT resolution, upsampled)
so row r belongs to channel r // 20 and frequency 1 + 2 * (r % 20) Hz. This
1-to-1 mapping lets Grad-CAM / occlusion maps be read back as
"which electrode, which rhythm, when".
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

FS = 128
N_CH, F_ROWS, T_COLS = 18, 20, 128
FREQS = [1 + 2 * i for i in range(F_ROWS)]  # lower edge of each 2 Hz row


def eeg_to_image(x: torch.Tensor) -> torch.Tensor:
    """x (B, 18, 1024) microvolts -> (B, 3, 360, 128) normalised log-spectrogram."""
    B, C, T = x.shape
    win = torch.hann_window(128, device=x.device)
    S = torch.stft(x.reshape(B * C, T).float(), n_fft=128, hop_length=16, window=win,
                   return_complex=True, center=False).abs() ** 2       # (B*C, 65, frames)
    S = S[:, 1:41]                                                     # 1..40 Hz
    S = S.reshape(B * C, F_ROWS, 2, -1).mean(2)                       # 2 Hz rows
    S = torch.log10(S + 1e-3)
    S = S.reshape(B, C * F_ROWS, -1)
    S = (S - S.mean((1, 2), keepdim=True)) / (S.std((1, 2), keepdim=True) + 1e-6)
    S = F.interpolate(S[:, None], size=(C * F_ROWS, T_COLS), mode="bilinear", align_corners=False)
    return S.expand(-1, 3, -1, -1).contiguous()


def resnet18(pretrained: bool) -> nn.Module:
    from torchvision.models import ResNet18_Weights, resnet18 as rn
    m = rn(weights=ResNet18_Weights.DEFAULT if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, 1)
    return m


class GradCAM:
    """Minimal Grad-CAM on a chosen layer (returns map at input resolution)."""

    def __init__(self, model: nn.Module, layer: nn.Module):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0]))

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        self.model.zero_grad()
        logit = self.model(img).squeeze(-1)
        logit.sum().backward()
        w = self.grads.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=img.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cam = cam / (cam.amax((1, 2), keepdim=True) + 1e-8)
        return cam.detach()

In [ ]:
%%writefile nb3_run.py
"""EEG-as-image seizure detection: ImageNet transfer learning + explainability, all CHB-MIT subjects.

Variants
  resnet18_imagenet      ImageNet ResNet-18, full fine-tuning
  efficientnet_b0_imagenet  ImageNet EfficientNet-B0, full fine-tuning
  resnet18_scratch       ResNet-18 random init (does natural-image pre-training help?)
  linear_probe           frozen ImageNet ResNet-18 features + logistic regression
Subject-wise k-fold; 2 training subjects form the validation set (best epoch, early stopping).
XAI (Grad-CAM + electrode / frequency occlusion) for resnet18_imagenet on every fold.
"""
from __future__ import annotations

import argparse
import copy
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression

from eeg_image import F_ROWS, N_CH, GradCAM, eeg_to_image, resnet18
from eegkit import make_windows, subject_folds, window_metrics

DEV = "cuda"
VARIANTS = ["resnet18_imagenet", "efficientnet_b0_imagenet", "resnet18_scratch", "linear_probe"]


def build(variant):
    if variant.startswith("resnet18"):
        return resnet18(variant.endswith("imagenet"))
    from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    return m


def batches(X, idx, bs):
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        yield j, torch.from_numpy(X[j]).to(DEV).float()


@torch.no_grad()
def logits(model, X, idx, bs=256, occlude=None):
    model.eval(); out = []
    for _, xb in batches(X, idx, bs):
        img = eeg_to_image(xb)
        if occlude is not None:
            img = occlude(img)
        with torch.autocast(DEV, dtype=torch.float16):
            o = model(img).squeeze(-1).float()
        if not torch.isfinite(o).all():      # fp16 overflow safety net: redo batch in fp32
            o = model(img.float()).squeeze(-1).float()
        out.append(o.cpu().numpy())
    return np.concatenate(out)


def train(variant, X, y, tr, va, epochs, patience, bs=64, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = build(variant).to(DEV).to(memory_format=torch.channels_last)
    lr = 1e-3 if variant.endswith("scratch") else 3e-4
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = max(20, epochs * int(np.ceil(len(tr) / bs)))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler()
    pos_w = torch.tensor((y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1), device=DEV)
    yt = torch.from_numpy(y.astype(np.float32)).to(DEV)
    best, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        for j, xb in batches(X, rng.permutation(tr), bs):
            img = eeg_to_image(xb)
            img = torch.roll(img, int(rng.integers(-16, 17)), dims=-1)        # time shift
            t0 = int(rng.integers(0, img.shape[-1] - 12)); img[..., t0:t0 + 12] = 0   # time mask
            f0 = int(rng.integers(0, F_ROWS - 2))                              # frequency mask
            img[:, :, [c * F_ROWS + f0 + k for c in range(N_CH) for k in range(2)]] = 0
            with torch.autocast(DEV, dtype=torch.float16):
                loss = F.binary_cross_entropy_with_logits(
                    model(img.contiguous(memory_format=torch.channels_last)).squeeze(-1), yt[j], pos_weight=pos_w)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        v = window_metrics(y[va], logits(model, X, va))["auprc"]
        if v > best:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
        print(f"    epoch {ep + 1:2d}  val AUPRC {v:.3f} (best {best:.3f})", flush=True)
        if bad >= patience:
            break
    model.load_state_dict(best_state)
    return model, best


@torch.no_grad()
def embed(model, X, idx, bs=256):
    model.eval(); fc = model.fc; model.fc = nn.Identity(); out = []
    for _, xb in batches(X, idx, bs):
        with torch.autocast(DEV, dtype=torch.float16):
            out.append(model(eeg_to_image(xb)).float().cpu().numpy())
    model.fc = fc
    return np.concatenate(out)


def explain(model, X, idx, n_examples=4):
    model = model.to(memory_format=torch.contiguous_format)
    cam_engine = GradCAM(model, model.layer3)
    cams = []
    for _, xb in batches(X, idx, 32):
        cams.append(cam_engine(eeg_to_image(xb)).cpu().numpy())
    cams = np.concatenate(cams)
    base = logits(model, X, idx)
    ch_drop, f_drop = [], []
    for c in range(N_CH):
        def occ(img, c=c):
            img = img.clone(); img[:, :, c * F_ROWS:(c + 1) * F_ROWS] = 0; return img
        ch_drop.append(float((base - logits(model, X, idx, occlude=occ)).mean()))
    for f in range(F_ROWS):
        rows = [c * F_ROWS + f for c in range(N_CH)]
        def occ(img, rows=rows):
            img = img.clone(); img[:, :, rows] = 0; return img
        f_drop.append(float((base - logits(model, X, idx, occlude=occ)).mean()))
    ex = np.argsort(-base)[:n_examples]
    with torch.no_grad():
        ex_img = eeg_to_image(torch.from_numpy(X[idx[ex]]).to(DEV).float())[:, 0].cpu().numpy()
    return {"cam_mean": cams.mean(0), "ch_drop": np.array(ch_drop), "f_drop": np.array(f_drop),
            "ex_img": ex_img, "ex_cam": cams[ex]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", required=True)
    ap.add_argument("--out", required=True, help="results dir (Drive)")
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--patience", type=int, default=5)
    ap.add_argument("--variants", default=",".join(VARIANTS))
    ap.add_argument("--inter_ratio", type=float, default=10.0)
    ap.add_argument("--xai_max", type=int, default=2000, help="max ictal windows explained per fold")
    ap.add_argument("--max_hours", type=float, default=0, help="stop starting new runs after this many hours (0 = no limit)")
    a = ap.parse_args()
    out = Path(a.out); (out / "xai").mkdir(parents=True, exist_ok=True)
    t_start = time.time()
    # cuDNN autotuning picks fp16 kernels that return NaN on GTX 16xx cards; safe on T4/A100/L4
    torch.backends.cudnn.benchmark = "GTX 16" not in torch.cuda.get_device_name()

    X, y, sub, _ = make_windows(Path(a.cache), win_s=8.0, ictal_stride_s=2.0, inter_stride_s=16.0,
                                inter_ratio=a.inter_ratio)
    subjects = sorted(set(sub))
    print(f"windows {X.shape} ictal {int(y.sum())} subjects {len(subjects)}", flush=True)
    folds = subject_folds(sub, y, k=a.folds if a.folds > 0 else len(subjects))
    rf = out / "results.json"
    res = json.loads(rf.read_text()) if rf.exists() else {}

    for fi, test_subs in enumerate(folds):
        te = np.where(np.isin(sub, test_subs))[0]
        rest = [s for s in subjects if s not in test_subs]
        val_subs = list(np.random.default_rng(fi).permutation(rest)[:2 if len(rest) >= 6 else 1])
        va = np.where(np.isin(sub, val_subs))[0]
        tr = np.setdiff1d(np.arange(len(y)), np.concatenate([te, va]))
        for variant in a.variants.split(","):
            key = f"{variant}|fold{fi}"
            if key in res:
                continue
            if a.max_hours and (time.time() - t_start) / 3600 > a.max_hours:
                print(f"time budget of {a.max_hours} h reached - stopping; finished runs are saved in {rf}", flush=True)
                return
            t0 = time.time()
            print(f"[fold {fi + 1}/{len(folds)} test={test_subs}] {variant}", flush=True)
            if variant == "linear_probe":
                m = resnet18(True).to(DEV)
                Etr, Ete = embed(m, X, tr), embed(m, X, te)
                mu, sd = Etr.mean(0), Etr.std(0) + 1e-6
                clf = LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000).fit((Etr - mu) / sd, y[tr])
                s = clf.decision_function((Ete - mu) / sd); best = None
            else:
                m, best = train(variant, X, y, tr, va, a.epochs, a.patience)
                s = logits(m, X, te)
                if variant == "resnet18_imagenet":
                    ict = te[y[te] == 1]
                    ict = np.random.default_rng(0).choice(ict, min(len(ict), a.xai_max), replace=False)
                    np.savez_compressed(out / "xai" / f"fold{fi}.npz", **explain(m, X, np.sort(ict)))
            r = window_metrics(y[te], s)
            r.update({"test_subjects": list(test_subs), "val_auprc": best,
                      "per_subject": {p: window_metrics(y[te][sub[te] == p], s[sub[te] == p])
                                      for p in test_subs if y[te][sub[te] == p].any()}})
            res[key] = r
            rf.write_text(json.dumps(res, indent=1))
            print(f"  -> AUROC {r['auroc']:.3f} AUPRC {r['auprc']:.3f} sens@95spec {r['sens_at_spec95']:.3f} "
                  f"({(time.time() - t0) / 60:.1f} min)", flush=True)
            del m; torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile figs.py
"""Tables (markdown) and figures for the three notebooks.  Usage: python figs.py nb1|nb2|nb3 RESULTS_DIR"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})


# ------------------------------------------------------------------------- NB1
def nb1(d: Path):
    R = json.loads((d / "finetune_results.json").read_text())
    names = {"cnn_scratch": "CNN", "cnn_robust": "CNN + robust aug.", "tf_scratch": "Transformer (no pre-train)",
             "tf_robust": "Transformer + robust aug.", "fm_finetune": "Foundation model",
             "fm_robust": "Foundation model + robust aug."}
    variants = [v for v in names if any(k.startswith(v + "|") for k in R)]
    conds = [c for c in next(iter(R.values())) if isinstance(next(iter(R.values()))[c], dict)
             and "auroc" in next(iter(R.values()))[c]]

    def stat(v, c, m):
        xs = [r[c][m] for k, r in R.items() if k.startswith(v + "|")]
        return np.mean(xs), np.std(xs), len(xs)

    md = []
    for m in ("auroc", "auprc", "sens_at_spec95"):
        md += [f"### {m.upper()} (mean ± s.d. over subject-wise folds)", "",
               "| condition | " + " | ".join(names[v] for v in variants) + " |", "|---|" + "---|" * len(variants)]
        for c in conds:
            vals = [stat(v, c, m) for v in variants]
            best = max(x[0] for x in vals)
            md.append(f"| {c} | " + " | ".join((f"**{a:.3f} ± {b:.3f}**" if a == best else f"{a:.3f} ± {b:.3f}")
                                               for a, b, _ in vals) + " |")
        md.append("")
    (d / "tables.md").write_text("\n".join(md)); print("\n".join(md))
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for v in variants:
        ax[0].plot([0, 25, 50, 75], [stat(v, "clean" if f == 0 else f"missing_{f}%", "auroc")[0]
                                     for f in (0, 25, 50, 75)], "o-", label=names[v])
    ax[0].set_xlabel("electrodes missing at test time (%)"); ax[0].set_ylabel("AUROC (unseen subjects)")
    ax[0].legend(frameon=False, fontsize=8)
    arts = ["clean", "flat_3ch_unflagged", "gauss_5dB", "gauss_0dB", "emg_bursts", "eye_blinks", "line_50Hz"]
    w = 0.8 / len(variants)
    for i, v in enumerate(variants):
        ax[1].bar(np.arange(len(arts)) + i * w, [stat(v, a, "auroc")[0] for a in arts], w, label=names[v])
    ax[1].set_xticks(np.arange(len(arts)) + 0.4 - w / 2)
    ax[1].set_xticklabels(["clean", "3 flat ch", "noise 5dB", "noise 0dB", "EMG*", "blinks*", "50Hz*"], rotation=30)
    ax[1].set_ylim(0.5, 1); ax[1].set_title("* artifact type never seen in training", fontsize=9)
    fig.tight_layout(); fig.savefig(d / "robustness.png", dpi=160)
    h = json.loads((d / "pretrain_history.json").read_text())
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.plot([e["epoch"] for e in h], [e["val_rel"] for e in h], "o-", ms=3)
    ax.set_xlabel("pre-training epoch (Siena)"); ax.set_ylabel("val MSE / mean-predictor MSE")
    fig.tight_layout(); fig.savefig(d / "pretraining_curve.png", dpi=160)


# ------------------------------------------------------------------------- NB2
def nb2(d: Path):
    R = json.loads((d / "results.json").read_text())
    hdr = ("| config | event sens. (%) | FA/h | median latency (s) | AUROC | ops/decision | energy (nJ) |\n"
           "|---|---|---|---|---|---|---|")
    md = []
    for g, title in (("A_models", "A. Classifiers (18 ch, fp32)"), ("B_channels", "B. Channel budget (logreg)"),
                     ("C_bits", "C. Fixed-point features + weights (logreg)"), ("D_adc", "D. ADC resolution (logreg, 18 ch)")):
        if g not in R:
            continue
        md += [f"### {title}", hdr]
        for tag, r in R[g].items():
            md.append(f"| {tag} | {100 * r['sensitivity']:.1f} | {r['fa_per_h']:.3f} | {r['median_latency_s']:.1f} | "
                      f"{r['mean_auroc']:.3f} | {r['cost']['ops'] / 1e3:.0f}k | {r['cost']['energy_nJ']:.0f} |")
        md.append("")
    r0 = R["B_channels"]["18ch"]
    md += [f"Total: {r0['n_seizures']} seizures, {r0['hours']:.0f} h of seizure-free EEG evaluated.", "",
           "### Per subject (logreg, 18 ch)", "| subject | seizures | sens. | FA/h | AUROC |", "|---|---|---|---|---|"]
    md += [f"| {s} | {v['n_seizures']} | {100 * v['sens']:.0f}% | {v['fa_per_h']:.2f} | "
           f"{v['auroc'] if v['auroc'] is None else round(v['auroc'], 3)} |" for s, v in r0["per_subject"].items()]
    (d / "tables.md").write_text("\n".join(md)); print("\n".join(md))
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for g, mk in (("B_channels", "o"), ("C_bits", "s")):
        rs = list(R[g].items())
        e = [r["cost"]["energy_nJ"] for _, r in rs]
        ax[0].scatter(e, [100 * r["sensitivity"] for _, r in rs], marker=mk, label=g[2:])
        ax[1].scatter(e, [r["fa_per_h"] for _, r in rs], marker=mk, label=g[2:])
        for (tag, r), x in zip(rs, e):
            ax[1].annotate(tag, (x, r["fa_per_h"]), fontsize=6, xytext=(2, 2), textcoords="offset points")
    for a in ax:
        a.set_xscale("log"); a.set_xlabel("energy per 4-s decision (nJ, 45 nm estimate)")
    ax[0].set_ylabel("event sensitivity (%)"); ax[1].set_ylabel("false alarms / hour"); ax[0].legend(frameon=False)
    fig.tight_layout(); fig.savefig(d / "pareto_energy.png", dpi=160)


# ------------------------------------------------------------------------- NB3
def nb3(d: Path):
    from eeg_image import F_ROWS, FREQS, N_CH
    from eegkit import MONTAGE
    R = json.loads((d / "results.json").read_text())
    names = {"resnet18_imagenet": "ResNet-18 (ImageNet, fine-tuned)",
             "efficientnet_b0_imagenet": "EfficientNet-B0 (ImageNet, fine-tuned)",
             "resnet18_scratch": "ResNet-18 (random init)", "linear_probe": "Frozen ImageNet ResNet-18 + LR"}
    md = ["| model | AUROC | AUPRC | sens. @ 95% spec. | folds |", "|---|---|---|---|---|"]
    for v, n in names.items():
        rs = [r for k, r in R.items() if k.startswith(v + "|")]
        if rs:
            f = lambda m: f"{np.mean([r[m] for r in rs]):.3f} ± {np.std([r[m] for r in rs]):.3f}"  # noqa: E731
            md.append(f"| {n} | {f('auroc')} | {f('auprc')} | {f('sens_at_spec95')} | {len(rs)} |")
    (d / "tables.md").write_text("\n".join(md)); print("\n".join(md))
    xs = [np.load(p) for p in sorted((d / "xai").glob("fold*.npz"))]
    if not xs:
        return
    ch = np.stack([x["ch_drop"] for x in xs]); fq = np.stack([x["f_drop"] for x in xs])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.4, 1]})
    ax[0].bar(MONTAGE, ch.mean(0), yerr=ch.std(0), color="#4a7")
    ax[0].tick_params(axis="x", rotation=90); ax[0].set_ylabel("logit drop when electrode occluded")
    ax[1].plot([f + 1 for f in FREQS], fq.T, alpha=0.4); ax[1].plot([f + 1 for f in FREQS], fq.mean(0), "k", lw=2.5)
    ax[1].set_xlabel("frequency (Hz)"); ax[1].set_ylabel("logit drop when band occluded")
    fig.tight_layout(); fig.savefig(d / "occlusion_importance.png", dpi=160)
    fig, ax = plt.subplots(2, len(xs), figsize=(2.3 * len(xs), 6.5), squeeze=False)
    for j, x in enumerate(xs):
        ax[0, j].imshow(x["ex_img"][0], aspect="auto", cmap="magma")
        ax[1, j].imshow(x["ex_img"][0], aspect="auto", cmap="gray")
        ax[1, j].imshow(x["ex_cam"][0], aspect="auto", cmap="jet", alpha=0.45)
        for a in ax[:, j]:
            a.set_yticks(np.arange(N_CH) * F_ROWS + F_ROWS / 2); a.set_yticklabels(MONTAGE if j == 0 else [], fontsize=5)
            a.set_xticks([])
        ax[0, j].set_title(f"fold {j}")
    fig.tight_layout(); fig.savefig(d / "gradcam_examples.png", dpi=160)
    cam = np.stack([x["cam_mean"].mean(1).reshape(N_CH, F_ROWS) for x in xs]).mean(0)
    fig, ax = plt.subplots(figsize=(5, 4.2))
    im = ax.imshow(cam, aspect="auto"); ax.set_yticks(range(N_CH)); ax.set_yticklabels(MONTAGE, fontsize=7)
    ax.set_xticks(range(0, F_ROWS, 4)); ax.set_xticklabels([FREQS[i] for i in range(0, F_ROWS, 4)])
    ax.set_xlabel("frequency (Hz)"); fig.colorbar(im); fig.tight_layout(); fig.savefig(d / "gradcam_channel_freq.png", dpi=160)


if __name__ == "__main__":
    {"nb1": nb1, "nb2": nb2, "nb3": nb3}[sys.argv[1]](Path(sys.argv[2]))

## 3. Data

In [ ]:
# ---- Data: download + preprocess CHB-MIT (all 24 cases: every seizure recording + 3 seizure-free per case) ----
# ~9 GB download, processed on the fly (raw EDFs deleted). Takes ~30-60 min.
import eegkit as K
CHB_CACHE = WORK / 'chb_cache'
if not (CHB_CACHE / 'index.json').exists():
    cat = K.select_for_windows(K.chbmit_catalog(WORK / 'raw'), n_free_per_case=3)
    print(len(cat), 'recordings selected,', sum(len(r['seizures']) for r in cat), 'seizures')
    K.build_cache(cat, WORK / 'raw', CHB_CACHE, workers=4)
idx = K.load_index(CHB_CACHE)
print(f"{len(idx)} recordings, {sum(r['n_samples'] for r in idx)/128/3600:.0f} h, "
      f"{sum(len(r['seizures']) for r in idx)} seizures, {len(set(r['subject'] for r in idx))} subjects")

## 4. Training + XAI (subject-wise 5-fold CV, stops cleanly after 10.5 h)

In [ ]:
!python nb3_run.py --cache {CHB_CACHE} --out {OUT} --folds 5 --epochs 20 --patience 5 --inter_ratio 10 --max_hours 10.5 --variants resnet18_imagenet,efficientnet_b0_imagenet,resnet18_scratch,linear_probe

In [ ]:
# ---- Results: tables + figures ----
!python figs.py nb3 {OUT}
from IPython.display import Markdown, Image, display
display(Markdown((OUT / 'tables.md').read_text()))
for f in sorted(OUT.glob('*.png')):
    print(f.name); display(Image(str(f)))

In [ ]:
import shutil
z = shutil.make_archive('/kaggle/working/project3_eeg_vision_xai_v2_results', 'zip', OUT)
print('download from the Output tab:', z)